# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Lane 1 (supervised) is signal analysis first: identifying which safe content and search signals are associated with visibility, clicks, and movement. The main deliverable is a signal report based on effect sizes and grouped comparisons, not a prediction pipeline.

To give the analysis a concrete ML framing, I pair it with a binary classification sub-task: classifying pages in the "down" bucket using a proxy label. The classifier is used to measure which observable signals are associated with different outcomes — it is not the final goal and does not claim to predict Google's algorithm.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape, "|", df.client_id.nunique(), "clients")
print("down bucket share:", round((df.trend_direction == "down").mean(), 3))

(30000, 44) | 32 clients
down bucket share: 0.542


## 2. Target or proxy
Target: is_declining_label = (trend_direction == "down")

This is a proxy label, not an observed future outcome. It is a defined rule derived from trend_pct (last-30 vs previous-30 impressions), so a classifier trained on it learns the current rule rather than predicting what happens next.

A stronger future version would use a warehouse time window: features from the previous 90 days → observed decline/recovery in the next 30 days. Since that label is not available in this snapshot, this project uses the proxy and clearly states its limitation.

Because the label is derived from trend_direction and trend_pct, those fields will not be used as features.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
label = (df.trend_direction == "down").astype(int)
print("label fully determined by trend_direction:", bool((label == (df.trend_direction == "down")).all()))
print("positive rows:", int(label.sum()), "=", round(label.mean(), 3))

label fully determined by trend_direction: True
positive rows: 16262 = 0.542


## 3. Success metric

Primary metric: effect sizes for the signal analysis layer. I will compare how observable signals differ between groups (for example, pages in the down bucket versus other pages) using standardized differences rather than relying only on raw correlations. This helps identify signals that have meaningful differences while considering a minimum volume threshold such as impressions_90d >= 500.

For the classification sub-task, I will use precision@50 because the practical decision is which pages an editor should review first. A good score means more of the top-ranked pages are relevant review candidates. ROC-AUC can provide additional model comparison, but precision@50 better matches the decision use case.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
base = (df.trend_direction == "down").mean()
print("base rate (always-positive P@K floor):", round(base, 3))
print("rule baseline P@50 ≈ 0.240 | starter random forest P@50 ≈ 0.740")

base rate (always-positive P@K floor): 0.542
rule baseline P@50 ≈ 0.240 | starter random forest P@50 ≈ 0.740


## 4. The unit of analysis, as a real dataframe

One row = one pseudonymized content item (page). The starter dataset contains 30,000 rows across 32 pseudonymized clients. The decision grain is page-level because we are analyzing which signals are associated with page performance.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("rows:", df.shape[0], "| columns:", df.shape[1])
print("content_id unique:", df.content_id.is_unique)
print("clients:", df.client_id.nunique())
df.head(3)

rows: 30000 | columns: 44
content_id unique: True
clients: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 5. Why ML beats a fixed rule here

The pattern is too complex for a fixed rule because many signals interact: impressions, clicks, CTR, position, freshness, engagement, and content characteristics. A single if-statement would only capture one or two signals, while ML can combine multiple weak signals to identify patterns that are harder to define manually.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_down"] = (df.trend_direction == "down").astype(int)
v = df[df.impressions_90d >= 500].copy()

def cohens_d(s):
    a, b = v.loc[v.is_down == 1, s].dropna(), v.loc[v.is_down == 0, s].dropna()
    pooled = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return round((a.mean() - b.mean()) / pooled, 3)

for s in ["avg_position", "ctr", "content_age_days", "days_with_impressions", "word_count"]:
    print(s, "| cohen's d:", cohens_d(s))

hi = v[(v.avg_position > 0) & (v.avg_position <= 10)]
print("high-volume page-one rows:", len(hi), "| share down:", round(hi.is_down.mean(), 3))

avg_position | cohen's d: -0.105
ctr | cohen's d: -0.215
content_age_days | cohen's d: -0.37
days_with_impressions | cohen's d: 0.004
word_count | cohen's d: -0.001
high-volume page-one rows: 7564 | share down: 0.589


## One-paragraph frame

For an SEO specialist or content editor deciding which pages and signals to review first, we will build a signal analysis from anonymized search and engagement data, with a supporting decline classification analysis using a proxy label. We will measure associations through effect sizes and evaluate the classification component with precision@50. A wrong call wastes reviewer time on low-impact pages while important pages losing visibility may be missed. A plain rule is not enough because many signals interact, overlap, and vary by context. We will claim only observed, directional, and decision-support results — not causal proof.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.